## 4. Hydraulic conductivity

### Set up hydraulic-conductivity inputs

Defines input paths, output workspace, constants, and basic environment settings for the surficial geology to HK workflow.

In [1]:
import os
import arcpy
from arcpy.sa import Raster, SetNull

# Ensure Spatial Analyst is available and outputs can be overwritten
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

print("Libraries loaded. Environment set.")


# Ensure Spatial Analyst is available and outputs can be overwritten
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

print("Libraries loaded. Environment set.")

## 1. Define File Paths and Constants
#Here we define the locations of the raw shapefiles, the CSV containing the parameters, the target template grid, and our conductivity constants.

# ============================================================
# INPUTS
# ============================================================
# path file to geology shapefile with surficial units (must have "POLYID" column for joining with Excel)
surfacegeology = r"S:\Data\Other_Data\Xu_2021\Data\09_Integrated_Surficial_Geology_Map\Integrated_surficial_geology_mapping.shp"
# path to calibrated HK values for surficial units (must have "unit_code" column matching the geology shapefile)
excel_path = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\Calibrated_surficial_Kh.xlsx"
in_sheet = rf"{excel_path}\GLB_surf_dissolve_merged$"
# extended model boundary with 2000 m buffer (for sampling HK beyond the lake bands)
extended_Buffer2km = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_2000m.shp"
# IBOUND raster for 2000 meter buffer boundary (1=active, 0=inactive,-1=lake) 
water_mask_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif"
# template raster to define the output grid and alignment (must have same CRS and resolution as model grid)
template_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"
Lakes_shp = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp"

MI_KRIG_UP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\UP_hkBulk_3174.tif"
MI_KRIG_LP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\LP_hkBulk_3174.tif"


# OUTPUT DIRECTORY
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK"
os.makedirs(out_dir, exist_ok=True)
gdb = os.path.join(out_dir, "HK_join_clip.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(gdb))

# ============================================================
# CONSTANTS & VALUES
# ============================================================
SEC_PER_DAY = 86400.0
LAKE_RASTER_VALUE = -1

# Kh Values (m/s)
LOWER_DEFAULT_MS = 1e-7      
FRAC_BEDROCK_MS  = 1e-6
BEDROCK_MS       = 1e-10

# Kh Values (m/day)
FRAC_BEDROCK_MDAY = FRAC_BEDROCK_MS * SEC_PER_DAY # 0.0864
BEDROCK_MDAY      = BEDROCK_MS * SEC_PER_DAY      # 0.0000864

# Values to burn into Lake / Land cells
LAKE_VALUE = 100.0
LAND_VALUE = 0.0

print("Paths and Constants Defined.")


Libraries loaded. Environment set.
Libraries loaded. Environment set.
Paths and Constants Defined.


### Helper functions for joins and geometry operations

Provides reusable helpers for deleting temporary data, making join fields, clipping, and erasing. These support the HK polygon processing workflow.

In [2]:
def delete_if_exists(p):
    if p and arcpy.Exists(p):
        arcpy.management.Delete(p)

def ensure_field(dataset, name, ftype, length=None):
    existing = {f.name for f in arcpy.ListFields(dataset)}
    if name not in existing:
        if ftype.upper() == "TEXT" and length is not None:
            arcpy.management.AddField(dataset, name, ftype, field_length=length)
        else:
            arcpy.management.AddField(dataset, name, ftype)

def make_join_id(dataset, source_field, join_field="JOIN_ID"):
    ensure_field(dataset, join_field, "TEXT", length=50)
    codeblock = """def to_key(v):
    if v is None: return None
    s = str(v).strip()
    if s == "": return None
    try:
        fv = float(s)
        if fv.is_integer(): s = str(int(fv))
        else: s = str(fv)
    except: pass
    return s.strip().upper()"""
    arcpy.management.CalculateField(dataset, join_field, f"to_key(!{source_field}!)", "PYTHON3", code_block=codeblock)
    
def safe_clip(in_fc, clip_fc, out_fc):
    delete_if_exists(out_fc)
    if hasattr(arcpy.analysis, "PairwiseClip"): arcpy.analysis.PairwiseClip(in_fc, clip_fc, out_fc)
    else: arcpy.analysis.Clip(in_fc, clip_fc, out_fc)

def safe_erase(in_fc, erase_fc, out_fc):
    delete_if_exists(out_fc)
    if hasattr(arcpy.analysis, "PairwiseErase"): arcpy.analysis.PairwiseErase(in_fc, erase_fc, out_fc)
    else: arcpy.analysis.Erase(in_fc, erase_fc, out_fc)

### Copy geology data and join calibrated HK attributes

Copies the source geology layer into a geodatabase, imports the calibrated HK CSV, standardizes join keys, and joins the tabular conductivity attributes to the geology polygons.
mathching columns of Zone* from calibrated HK and Polygon_ID 

In [3]:
# output feature class paths
surf_fc = os.path.join(gdb, "surfacegeology_base")
csv_tbl = os.path.join(gdb, "Kh_csv_tbl")
joined_fc = os.path.join(gdb, "surfacegeo_joined")


print("Copying Geology to GDB...")
delete_if_exists(surf_fc)
arcpy.management.CopyFeatures(surfacegeology, surf_fc)

print("Importing CSV to GDB...")

delete_if_exists(csv_tbl)
arcpy.conversion.TableToTable(in_sheet, gdb, "Kh_csv_tbl")

tbl_fields = [f.name for f in arcpy.ListFields(csv_tbl)]
if "Lower_ms" not in tbl_fields:
    ensure_field(csv_tbl, "Lower_ms", "DOUBLE")
    arcpy.management.CalculateField(csv_tbl, "Lower_ms", str(LOWER_DEFAULT_MS), "PYTHON3")

print("Processing Join Keys...")
# Note: Ensure that "col_1" is your actual ID field in the CSV matching "POLYID"
make_join_id(surf_fc, "POLYID", "JOIN_ID")
make_join_id(csv_tbl, "Zone_", "JOIN_ID")

print("Joining Data...")
delete_if_exists(joined_fc)
arcpy.management.CopyFeatures(surf_fc, joined_fc)
csv_fields_to_add = [f.name for f in arcpy.ListFields(csv_tbl) if f.type not in ("OID", "Geometry") and f.name.lower() != "join_id"]
arcpy.management.JoinField(joined_fc, "JOIN_ID", csv_tbl, "JOIN_ID", csv_fields_to_add)

print(f"✅ Join Complete. Added {len(csv_fields_to_add)} attributes.")


Copying Geology to GDB...
Importing CSV to GDB...
Processing Join Keys...
Joining Data...
✅ Join Complete. Added 6 attributes.


### Calculate five MODFLOW HK layers

Calculates the final five hydraulic-conductivity fields in m/day for the MODFLOW layers: upper, middle, lower, fractured bedrock, and bedrock.

In [4]:
# The 5 HK fields required by MODFLOW
fields_5_layers = ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d", "FracKh_m_d", "BedKh_m_d"]

print("Checking joined_fc fields...")
joined_fields = [f.name for f in arcpy.ListFields(joined_fc)]
print(joined_fields)

print("Calculating 5-Layer HK fields in m/day...")
for fld in fields_5_layers:
    ensure_field(joined_fc, fld, "DOUBLE")

# Actual source field names in joined_fc
upper_src  = "Kh__m_s_"
middle_src = "Kh__m_s_1"
lower_src  = "Lower_ms"

missing = [f for f in [upper_src, middle_src, lower_src] if f not in joined_fields]
if missing:
    raise ValueError(f"Missing expected field(s) in joined_fc: {missing}")

codeblock = """
def to_float(v):
    try:
        if v is None:
            return None
        s = str(v).strip()
        if s == "":
            return None
        return float(s)
    except:
        return None
"""

# Surficial Soil Layers (m/s to m/day)
arcpy.management.CalculateField(
    joined_fc, "UPKh_m_d",
    f"to_float(!{upper_src}!) * {SEC_PER_DAY}",
    "PYTHON3", codeblock
)

arcpy.management.CalculateField(
    joined_fc, "MidKh_m_d",
    f"to_float(!{middle_src}!) * {SEC_PER_DAY}",
    "PYTHON3", codeblock
)

arcpy.management.CalculateField(
    joined_fc, "LowKh_m_d",
    f"to_float(!{lower_src}!) * {SEC_PER_DAY}",
    "PYTHON3", codeblock
)

# Deep Constant Bedrock Layers
arcpy.management.CalculateField(
    joined_fc, "FracKh_m_d",
    str(FRAC_BEDROCK_MDAY),
    "PYTHON3"
)

arcpy.management.CalculateField(
    joined_fc, "BedKh_m_d",
    str(BEDROCK_MDAY),
    "PYTHON3"
)

print("✅ 5 Stratigraphy Layers calculated.")

Checking joined_fc fields...
['OBJECTID', 'Shape', 'POLYID', 'JOIN_ID', 'Shape_Length', 'Shape_Area', 'Zone_', 'Polygon_ID', 'Description', 'Kh__m_s_', 'Kh__m_s_1', 'Lower_ms']
Calculating 5-Layer HK fields in m/day...
✅ 5 Stratigraphy Layers calculated.


### Clip HK polygons to the model domain and apply lake values

Clips the geology/HK polygons to the buffered model domain, extracts lake polygons from the water mask, splits land and lake portions, and assigns lake HK values where needed.

In [5]:
from arcpy.sa import Raster, SetNull, Int

print("Clipping to Domain Buffer...")

# ------------------------------------------------------------
# inputs / outputs
# ------------------------------------------------------------
domain_fc    = os.path.join(gdb, "domain_buffer2km_proj")
clip_fc      = os.path.join(gdb, "surfacegeo_joined_clip2km")

lake_ras     = os.path.join(gdb, "lake_ras")
lake_poly0   = os.path.join(gdb, "lake_raw")
lake_poly    = os.path.join(gdb, "lake_diss")
lake_domain  = os.path.join(gdb, "lake_in_domain")

lake_part    = os.path.join(gdb, "hk_lake_part")
land_part    = os.path.join(gdb, "hk_land_part")
lake_missing = os.path.join(gdb, "hk_lake_missing")

final_fc     = os.path.join(gdb, "surfacegeo_HK_FINAL")

# ------------------------------------------------------------
# make sure this matches your raster coding
# lake cells in water_mask_raster = -1
# ------------------------------------------------------------
LAKE_RASTER_VALUE = -1

# ------------------------------------------------------------
# project domain buffer if needed
# ------------------------------------------------------------
buf_sr = arcpy.Describe(extended_Buffer2km).spatialReference
fc_sr  = arcpy.Describe(joined_fc).spatialReference

delete_if_exists(domain_fc)
if fc_sr.factoryCode == buf_sr.factoryCode:
    arcpy.management.CopyFeatures(extended_Buffer2km, domain_fc)
else:
    arcpy.management.Project(extended_Buffer2km, domain_fc, fc_sr)

# ------------------------------------------------------------
# clip joined geology to domain
# ------------------------------------------------------------
delete_if_exists(clip_fc)
safe_clip(joined_fc, domain_fc, clip_fc)

# ------------------------------------------------------------
# extract lake polygons from raster mask
# keeps only cells == -1, makes them value 1, everything else NoData
# Int() makes sure RasterToPolygon gets an integer raster
# ------------------------------------------------------------
print("Extracting Lake boundaries from raster...")

for p in [lake_ras, lake_poly0, lake_poly]:
    delete_if_exists(p)

ras = Raster(water_mask_raster)
Int(SetNull(ras != LAKE_RASTER_VALUE, 1)).save(lake_ras)

arcpy.conversion.RasterToPolygon(lake_ras, lake_poly0, "SIMPLIFY", "VALUE")
arcpy.management.Dissolve(lake_poly0, lake_poly)

# ------------------------------------------------------------
# clip lake polygons to model domain
# ------------------------------------------------------------
print("Splitting Geometry to Land and Lakes...")

for p in [lake_domain, lake_part, land_part, lake_missing]:
    delete_if_exists(p)

safe_clip(lake_poly, domain_fc, lake_domain)

# parts:
# 1) geology that overlaps lake polygons
# 2) geology on land only
# 3) any lake area with no geology polygon coverage
safe_clip(clip_fc, lake_domain, lake_part)
safe_erase(clip_fc, lake_domain, land_part)
safe_erase(lake_domain, lake_part, lake_missing)

# ------------------------------------------------------------
# assign recharge / lake values
# ------------------------------------------------------------
print("Applying lake and land values...")

for fc in [lake_part, land_part, lake_missing]:
    ensure_field(fc, "Rech_m_d", "DOUBLE")

arcpy.management.CalculateField(land_part, "Rech_m_d", str(LAND_VALUE), "PYTHON3")
arcpy.management.CalculateField(lake_part, "Rech_m_d", str(LAKE_VALUE), "PYTHON3")
arcpy.management.CalculateField(lake_missing, "Rech_m_d", str(LAKE_VALUE), "PYTHON3")

# ------------------------------------------------------------
# force top 3 HK layers in lake areas to LAKE_VALUE
# keep fractured and deep bedrock unchanged
# ------------------------------------------------------------
for f in ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]:
    ensure_field(lake_part, f, "DOUBLE")
    ensure_field(lake_missing, f, "DOUBLE")
    arcpy.management.CalculateField(lake_part, f, str(LAKE_VALUE), "PYTHON3")
    arcpy.management.CalculateField(lake_missing, f, str(LAKE_VALUE), "PYTHON3")

# ------------------------------------------------------------
# FIX: explicitly fill fracture + bedrock in lake polygons too
# this avoids null gaps in lake_missing after merge
# ------------------------------------------------------------
for f, val in [("FracKh_m_d", FRAC_BEDROCK_MDAY), ("BedKh_m_d", BEDROCK_MDAY)]:
    ensure_field(lake_part, f, "DOUBLE")
    ensure_field(lake_missing, f, "DOUBLE")
    arcpy.management.CalculateField(lake_part, f, str(val), "PYTHON3")
    arcpy.management.CalculateField(lake_missing, f, str(val), "PYTHON3")

# ------------------------------------------------------------
# merge all parts back together
# ------------------------------------------------------------
delete_if_exists(final_fc)
arcpy.management.Merge([land_part, lake_part, lake_missing], final_fc)

print(f"✅ Geometry merging complete. Rows: {int(arcpy.management.GetCount(final_fc)[0])}")
print(f"Final output: {final_fc}")

Clipping to Domain Buffer...
Extracting Lake boundaries from raster...
Splitting Geometry to Land and Lakes...
Applying lake and land values...
✅ Geometry merging complete. Rows: 74
Final output: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_join_clip.gdb\surfacegeo_HK_FINAL


### Now in the part I want to do the followings 
1- add two columns of called KrigUP_HK and KrigMid_HK
2- Do a zonal stats of this final_fc = os.path.join(gdb, "surfacegeo_HK_FINAL") with these two rasters MIKRIG_UP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\UP_hkBulk_3174.tif"
MIKRIG_LP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\LP_hkBulk_3174.tif" and replace the UPKh_m_d of those overlapping area with the rasters in the KrigUP_HK column 


In [6]:
import os
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling

# -------------------------------
# INPUTS
# -------------------------------
MI_KRIG_UP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\UP_hkBulk_3174.tif"
MI_KRIG_LP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\LP_hkBulk_3174.tif"
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK"

out_merged = os.path.join(out_dir, "HKBUlk_MI_Krig.tif")

# make output folder if needed
os.makedirs(out_dir, exist_ok=True)

# -------------------------------
# OPEN INPUT RASTERS
# -------------------------------
src_files = [rasterio.open(MI_KRIG_UP), rasterio.open(MI_KRIG_LP)]

# -------------------------------
# MERGE AT 1000 m RESOLUTION
# -------------------------------
mosaic, out_transform = merge(
    src_files,
    res=(1000, 1000),
    method="first"   # keeps first raster values where overlap exists
)

# -------------------------------
# OUTPUT METADATA
# -------------------------------
out_meta = src_files[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "count": mosaic.shape[0],
    "crs": src_files[0].crs,
    "dtype": mosaic.dtype
})

# -------------------------------
# WRITE OUTPUT
# -------------------------------
with rasterio.open(out_merged, "w", **out_meta) as dest:
    dest.write(mosaic)

# close input rasters
for src in src_files:
    src.close()

print("Merged raster saved to:")
print(out_merged)

Merged raster saved to:
D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HKBUlk_MI_Krig.tif


In [8]:
MIKrig = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HKBUlk_MI_Krig.tif"

# HK Zonal Statistics — Geometric Mean (Log-Mean) Workflow

This notebook extracts hydraulic conductivity (HK) values from a Michigan Basin kriging raster and assigns them to Great Lakes Basin polygons using the **geometric mean** approach:

`UPKh_krig = 10^(mean(log10(HK)))` per polygon

---

## Cell 1 — Imports and Helper Functions

Import required libraries and define reusable utility functions for deleting existing datasets and ensuring fields exist on feature classes.

```python
import os
from arcpy.sa import Con, IsNull, Raster, Log10, Nibble
from collections import defaultdict, Counter

def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def ensure_field(fc, field_name, field_type):
    existing = [f.name for f in arcpy.ListFields(fc)]
    if field_name not in existing:
        arcpy.management.AddField(fc, field_name, field_type)
```

---

## Cell 2 — User Inputs

Set the paths to your input polygon, input raster, and output geodatabase. Update these before running.

```python
final_fc = r"path\to\your\polygon_feature_class"
MIKrig   = r"path\to\your\HK_kriging_raster"
gdb      = r"path\to\your\output.gdb"
```

---

## Cell 3 — Step 1: Reproject Polygon to EPSG:3174

Check the polygon's spatial reference. If it is not already in EPSG:3174 (Great Lakes Albers), reproject it. This ensures both the polygon and the raster share the same coordinate system for zonal statistics.

```python
sr_3174 = arcpy.SpatialReference(3174)

print("STEP 1: Reprojecting polygon to EPSG:3174...")

poly_sr = arcpy.Describe(final_fc).spatialReference
if poly_sr.factoryCode != 3174:
    print(f"  ⚠️  Polygon SR is {poly_sr.name} — reprojecting...")
    poly_3174 = os.path.join(gdb, "poly_3174")
    delete_if_exists(poly_3174)
    arcpy.management.Project(final_fc, poly_3174, sr_3174)
    print(f"  ✅ Polygon reprojected: {poly_3174}")
else:
    poly_3174 = final_fc
    print(f"  ✅ Polygon already in EPSG:3174.")
```

---

## Cell 4 — Step 2: Reproject Raster to EPSG:3174

Same check for the HK kriging raster. Uses bilinear interpolation for reprojection since the raster contains continuous values.

```python
print("\nSTEP 2: Reprojecting raster to EPSG:3174...")

ras_sr = arcpy.Describe(MIKrig).spatialReference
if ras_sr.factoryCode != 3174:
    print(f"  ⚠️  Raster SR is {ras_sr.name} — reprojecting...")
    raster_3174 = os.path.join(gdb, "MIKrig_3174")
    delete_if_exists(raster_3174)
    arcpy.management.ProjectRaster(
        MIKrig, raster_3174, sr_3174, "BILINEAR"
    )
    MIKrig_work = raster_3174
    print(f"  ✅ Raster reprojected: {raster_3174}")
else:
    MIKrig_work = MIKrig
    print(f"  ✅ Raster already in EPSG:3174.")
```

---

## Cell 5 — Step 3: Build Raster Domain Polygon

Create a polygon that represents the spatial extent of valid (non-NoData) raster cells. This polygon will be used to clip the input polygons so that only those overlapping the raster are processed.

```python
print("\nSTEP 3: Building raster domain polygon...")

ras_obj    = Raster(MIKrig_work)
valid_mask = Con(~IsNull(ras_obj), 1)

raster_domain_raw = os.path.join(gdb, "temp_domain_raw")
raster_domain     = os.path.join(gdb, "temp_domain")
delete_if_exists(raster_domain_raw)
delete_if_exists(raster_domain)

arcpy.conversion.RasterToPolygon(
    valid_mask, raster_domain_raw,
    "NO_SIMPLIFY", "Value", "SINGLE_OUTER_PART"
)

arcpy.management.Dissolve(raster_domain_raw, raster_domain)
delete_if_exists(raster_domain_raw)

print(f"  ✅ Raster domain polygon created.")
```

---

## Cell 6 — Add Unique Join Key (UNIQ_ID)

Add a `UNIQ_ID` field to the polygon layer, populated from `OBJECTID`. This key is set **before** clipping so it survives into the clipped feature class and can be used to join values back to the original polygons. Also store original polygon areas for sliver detection.

```python
print("\nAdding unique join key to polygon layer...")
ensure_field(poly_3174, "UNIQ_ID", "LONG")
arcpy.management.CalculateField(
    poly_3174, "UNIQ_ID",
    f"!{arcpy.Describe(poly_3174).OIDFieldName}!",
    "PYTHON3"
)
print(f"  ✅ UNIQ_ID populated from OBJECTID — guaranteed unique per row.")

uniq_counts = Counter()
with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID"]) as cur:
    for row in cur:
        uniq_counts[row[0]] += 1
print(f"  Total rows     : {sum(uniq_counts.values())}")
print(f"  Unique UNIQ_ID : {len(uniq_counts)}")
print(f"  Duplicates     : {sum(1 for c in uniq_counts.values() if c > 1)}")

# Store original areas for sliver detection later
orig_areas = {}
with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID", "SHAPE@AREA"]) as cur:
    for uniq_id, area in cur:
        orig_areas[uniq_id] = area
```

---

## Cell 7 — Step 4: Clip Polygons to Raster Domain + Remove Slivers

Clip the polygon layer to the raster domain. After clipping, remove **sliver fragments** — polygons whose clipped area is less than 5% of their original area. These are edge polygons that barely touch the raster and would incorrectly receive values. Only the largest fragment per `UNIQ_ID` is kept.

```python
print("\nSTEP 4: Clipping polygon to raster domain...")

clipped_fc_raw = os.path.join(gdb, "surfacegeo_HK_Krig_clipped_raw")
clipped_fc     = os.path.join(gdb, "surfacegeo_HK_Krig_clipped")
delete_if_exists(clipped_fc_raw)
delete_if_exists(clipped_fc)

arcpy.analysis.Clip(poly_3174, raster_domain, clipped_fc_raw)

raw_count   = int(arcpy.management.GetCount(clipped_fc_raw).getOutput(0))
total_count = int(arcpy.management.GetCount(poly_3174).getOutput(0))
print(f"  Raw clip result: {raw_count} features from {total_count} originals.")

# --- Remove slivers ---
print("  Removing sliver fragments...")

MIN_OVERLAP_FRACTION = 0.05  # 5% minimum overlap required

oid_field_raw = arcpy.Describe(clipped_fc_raw).OIDFieldName

uniq_fragments = defaultdict(list)
with arcpy.da.SearchCursor(
        clipped_fc_raw, [oid_field_raw, "UNIQ_ID", "SHAPE@AREA"]) as cur:
    for oid, uniq_id, area in cur:
        uniq_fragments[uniq_id].append((oid, area))

keep_oids = set()
removed_slivers = []
for uniq_id, fragments in uniq_fragments.items():
    total_clip_area = sum(a for _, a in fragments)
    orig_area = orig_areas.get(uniq_id, 0)

    if orig_area > 0 and (total_clip_area / orig_area) < MIN_OVERLAP_FRACTION:
        removed_slivers.append(uniq_id)
        continue

    best_oid = max(fragments, key=lambda x: x[1])[0]
    keep_oids.add(best_oid)

all_oids_raw = set()
with arcpy.da.SearchCursor(clipped_fc_raw, [oid_field_raw]) as cur:
    for row in cur:
        all_oids_raw.add(row[0])

remove_oids = all_oids_raw - keep_oids

if remove_oids:
    lyr = "clipped_raw_lyr"
    arcpy.management.MakeFeatureLayer(clipped_fc_raw, lyr)
    where = f"{oid_field_raw} IN ({','.join(str(o) for o in remove_oids)})"
    arcpy.management.SelectLayerByAttribute(lyr, "NEW_SELECTION", where)
    removed_count = int(arcpy.management.GetCount(lyr).getOutput(0))
    arcpy.management.DeleteRows(lyr)
    arcpy.management.Delete(lyr)
    print(f"  ✅ Removed {removed_count} sliver/tiny-overlap fragments.")
    if removed_slivers:
        print(f"     UNIQ_IDs removed (< {MIN_OVERLAP_FRACTION*100:.0f}% overlap): "
              f"{sorted(removed_slivers)}")

arcpy.management.CopyFeatures(clipped_fc_raw, clipped_fc)
delete_if_exists(clipped_fc_raw)

clipped_count = int(arcpy.management.GetCount(clipped_fc).getOutput(0))
print(f"  ✅ {clipped_count} of {total_count} polygons kept after sliver removal.")
print(f"     {total_count - clipped_count} polygons removed.")

clipped_fields = [f.name for f in arcpy.ListFields(clipped_fc)]
if "UNIQ_ID" not in clipped_fields:
    raise RuntimeError("UNIQ_ID was not carried through the clip — cannot join.")
print(f"  ✅ UNIQ_ID confirmed present in clipped FC.")

clipped_uniq_ids = set()
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID"]) as cur:
    for row in cur:
        clipped_uniq_ids.add(row[0])
print(f"  Unique UNIQ_IDs in clipped: {len(clipped_uniq_ids)}")
print(f"  Clipped UNIQ_IDs: {sorted(clipped_uniq_ids)}")

uniq_clipped = Counter()
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID"]) as cur:
    for row in cur:
        uniq_clipped[row[0]] += 1
dups = {k: v for k, v in uniq_clipped.items() if v > 1}
if dups:
    print(f"  ⚠️  Duplicate UNIQ_IDs in clipped (slivers): {sorted(dups.keys())}")
```

---

## Cell 8 — Step 5: Add Fields to Clipped Feature Class

Add `ZONE_ID` (for zonal statistics), `UPKh_krig`, and `MidKh_krig` fields to the clipped feature class. `ZONE_ID` is set from the new OBJECTID of the clipped FC (not the original OBJECTID).

```python
print("\nSTEP 5: Adding fields to clipped feature class...")

ensure_field(clipped_fc, "ZONE_ID",    "LONG")
ensure_field(clipped_fc, "UPKh_krig",  "DOUBLE")
ensure_field(clipped_fc, "MidKh_krig", "DOUBLE")

oid_field = arcpy.Describe(clipped_fc).OIDFieldName
arcpy.management.CalculateField(clipped_fc, "ZONE_ID", f"!{oid_field}!", "PYTHON3")

print(f"  ✅ ZONE_ID populated from {oid_field}.")
print(f"  ✅ UPKh_krig and MidKh_krig added.")
```

---

## Cell 9 — Step 5b: Create Log10-Transformed Raster

Create a new raster where each cell = `log10(HK)`. Only valid positive cells are transformed. This is used as input for zonal statistics so that the mean is computed in log-space, yielding the geometric mean after back-transformation.

```python
print("\nSTEP 5b: Creating log10-transformed raster...")

ras_work = Raster(MIKrig_work)
log_raster_path = os.path.join(gdb, "temp_log10_HK")
delete_if_exists(log_raster_path)

log_raster = Con(ras_work > 0, Log10(ras_work))
log_raster.save(log_raster_path)
print(f"  ✅ Log10(HK) raster created.")
```

---

## Cell 10 — Step 6: Zonal Statistics (MEAN on log10 HK)

Run Zonal Statistics as Table using `MEAN` on the log10-transformed raster. Each polygon zone gets the average of `log10(HK)` values within it.

```python
print("\nSTEP 6: Running Zonal Statistics (MEAN on log10 HK) on clipped polygons...")

arcpy.env.snapRaster = MIKrig_work
arcpy.env.cellSize   = MIKrig_work

zs_table = os.path.join(gdb, "temp_zs_krig")
delete_if_exists(zs_table)

try:
    arcpy.sa.ZonalStatisticsAsTable(
        clipped_fc, "ZONE_ID",
        log_raster_path, zs_table,
        "DATA", "MEAN"
    )
    if not arcpy.Exists(zs_table):
        raise RuntimeError("Output table not found after ZonalStatisticsAsTable.")
    zs_count = int(arcpy.management.GetCount(zs_table).getOutput(0))
    print(f"  ✅ Zonal Statistics complete — {zs_count} rows returned.")
except Exception as e:
    print(f"  ❌ ZonalStatisticsAsTable failed: {e}")
    print(f"     arcpy messages:\n{arcpy.GetMessages()}")
    raise

arcpy.env.snapRaster = None
arcpy.env.cellSize   = None
```

---

## Cell 11 — Step 7: Back-Transform and Write Geometric Mean

Read the `MEAN` field from the zonal statistics table. Back-transform each value: `UPKh_krig = 10^(mean(log10(HK)))` to get the geometric mean in real HK units. `MidKh_krig = UPKh_krig / 10`.

```python
print("\nSTEP 7: Writing geometric mean HK values to clipped FC...")

zs_fields = [f.name for f in arcpy.ListFields(zs_table)]
print(f"  ZonalStats table fields: {zs_fields}")

if "MEAN" not in zs_fields:
    raise RuntimeError(f"MEAN field not found. Available: {zs_fields}")

zs_dict = {}
with arcpy.da.SearchCursor(zs_table, ["ZONE_ID", "MEAN", "COUNT"]) as cur:
    for row in cur:
        zone_id, mean_log_val, count_val = row
        if mean_log_val is not None and count_val >= 1:
            zs_dict[zone_id] = 10 ** mean_log_val

print(f"  Valid ZonalStats results: {len(zs_dict)}")

updated = 0
with arcpy.da.UpdateCursor(
        clipped_fc, ["ZONE_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        zone_id = row[0]
        if zone_id in zs_dict:
            up_val = zs_dict[zone_id]
            row[1] = up_val
            row[2] = up_val / 10.0
            updated += 1
            cursor.updateRow(row)

print(f"  ✅ UPKh_krig (geometric mean) updated in {updated} clipped polygons.")
print(f"  ✅ MidKh_krig = UPKh_krig / 10 in {updated} clipped polygons.")
```

---

## Cell 12 — Step 7b: Fill NULL Polygons with Nearest Raster Value

Some clipped polygons may fall entirely on NoData cells and receive no value from zonal statistics. For these, use Nibble to expand the log10 raster into NoData gaps, sample at polygon centroids, and back-transform the result.

```python
print("\nSTEP 7b: Filling NULL UPKh_krig polygons...")

null_zone_ids = set()
with arcpy.da.SearchCursor(clipped_fc, ["ZONE_ID", "UPKh_krig"]) as cur:
    for row in cur:
        if row[1] is None:
            null_zone_ids.add(row[0])

print(f"  NULL polygons inside raster domain: {len(null_zone_ids)}")

if len(null_zone_ids) > 0:
    null_lyr = "null_polys_lyr"
    arcpy.management.MakeFeatureLayer(clipped_fc, null_lyr)
    null_ids_str = ", ".join(str(z) for z in null_zone_ids)
    arcpy.management.SelectLayerByAttribute(
        null_lyr, "NEW_SELECTION", f"ZONE_ID IN ({null_ids_str})")

    null_centroids = os.path.join(gdb, "temp_null_centroids")
    delete_if_exists(null_centroids)
    arcpy.management.FeatureToPoint(null_lyr, null_centroids, "INSIDE")
    arcpy.management.Delete(null_lyr)

    print(f"  ✅ {int(arcpy.management.GetCount(null_centroids).getOutput(0))} "
          f"NULL centroids created.")

    print("  Filling raster NoData holes with Nibble...")
    ras_log     = Raster(log_raster_path)
    valid_cells = Con(~IsNull(ras_log), 1)

    filled_raster = os.path.join(gdb, "temp_filled_raster")
    delete_if_exists(filled_raster)

    nibbled = arcpy.sa.Nibble(
        ras_log, valid_cells, "DATA_ONLY", "PROCESS_NODATA")
    nibbled.save(filled_raster)
    print(f"  ✅ Filled raster saved.")

    sample_null = os.path.join(gdb, "temp_sample_null")
    delete_if_exists(sample_null)
    arcpy.sa.Sample(
        filled_raster, null_centroids, sample_null, "NEAREST", "ZONE_ID")

    sample_fields = [f.name for f in arcpy.ListFields(sample_null)]
    link_field       = sample_fields[1]
    raster_val_field = sample_fields[-1]

    centroid_oid_field   = arcpy.Describe(null_centroids).OIDFieldName
    centroid_oid_to_zone = {}
    with arcpy.da.SearchCursor(null_centroids,
                               [centroid_oid_field, "ZONE_ID"]) as cur:
        for row in cur:
            centroid_oid_to_zone[row[0]] = row[1]

    filled_values = {}
    with arcpy.da.SearchCursor(sample_null,
                               [link_field, raster_val_field]) as cur:
        for row in cur:
            raw_oid, val = row
            zone_id = (
                centroid_oid_to_zone.get(raw_oid) or
                centroid_oid_to_zone.get(
                    int(raw_oid) if raw_oid is not None else None) or
                centroid_oid_to_zone.get(str(raw_oid))
            )
            if zone_id is not None and val is not None:
                filled_values[zone_id] = 10 ** val

    print(f"  ✅ Nearest values found for {len(filled_values)} of "
          f"{len(null_zone_ids)} NULL polygons.")

    fill_count = 0
    with arcpy.da.UpdateCursor(
            clipped_fc, ["ZONE_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
        for row in cursor:
            zone_id = row[0]
            if zone_id in filled_values:
                up_val = filled_values[zone_id]
                row[1] = up_val
                row[2] = up_val / 10.0
                fill_count += 1
                cursor.updateRow(row)

    print(f"  ✅ {fill_count} NULL polygons filled.")

    still_null = null_zone_ids - set(filled_values.keys())
    if still_null:
        print(f"  ⚠️  {len(still_null)} polygons could not be filled: "
              f"{sorted(still_null)}")

    delete_if_exists(null_centroids)
    delete_if_exists(sample_null)
    delete_if_exists(filled_raster)
else:
    print("  ✅ No NULL polygons — nothing to fill.")
```

---

## Cell 13 — Step 7c: Diagnostic Check

Print which `UNIQ_ID`s have values and which are still NULL in the clipped feature class. Use this to verify that only the correct Michigan polygons received HK values before joining back to the original layer.

```python
print("\n  --- DIAGNOSTIC: UNIQ_IDs with non-NULL UPKh_krig in clipped FC ---")
valued_uniq_ids = set()
null_uniq_ids   = set()
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID", "UPKh_krig"]) as cur:
    for row in cur:
        if row[1] is not None:
            valued_uniq_ids.add(row[0])
        else:
            null_uniq_ids.add(row[0])
print(f"  UNIQ_IDs with values : {sorted(valued_uniq_ids)}")
print(f"  UNIQ_IDs still NULL  : {sorted(null_uniq_ids)}")
```

---

## Cell 14 — Step 8: Join Values Back to Original Polygon

Transfer `UPKh_krig` and `MidKh_krig` from the clipped feature class back to the original polygon layer using `UNIQ_ID`. **Strict rule**: only polygons whose `UNIQ_ID` exists in the clipped FC with a non-NULL value receive data. **All other polygons are explicitly set to NULL** — no exceptions, no stale values.

```python
print("\nSTEP 8: Assigning values to original polygon using UNIQ_ID...")

ensure_field(poly_3174, "UPKh_krig",  "DOUBLE")
ensure_field(poly_3174, "MidKh_krig", "DOUBLE")

uniq_lookup = {}
sliver_ids  = []

with arcpy.da.SearchCursor(
        clipped_fc, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cur:
    for row in cur:
        uniq_id, up_val, mid_val = row
        if uniq_id is None or up_val is None:
            continue
        if uniq_id in uniq_lookup:
            sliver_ids.append(uniq_id)
            if up_val > uniq_lookup[uniq_id][0]:
                uniq_lookup[uniq_id] = (up_val, up_val / 10.0)
        else:
            uniq_lookup[uniq_id] = (up_val, up_val / 10.0)

print(f"  Unique UNIQ_IDs with values : {len(uniq_lookup)}")
print(f"  UNIQ_IDs getting values: {sorted(uniq_lookup.keys())}")
if sliver_ids:
    print(f"  Slivers resolved (MAX kept) : {sorted(set(sliver_ids))}")

updated = 0
nulled  = 0
with arcpy.da.UpdateCursor(
        poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if uniq_id in uniq_lookup:
            row[1] = uniq_lookup[uniq_id][0]
            row[2] = uniq_lookup[uniq_id][1]
            updated += 1
        else:
            row[1] = None
            row[2] = None
            nulled += 1
        cursor.updateRow(row)

print(f"  ✅ {updated} polygons received values (matched on UNIQ_ID).")
print(f"  ✅ {nulled} polygons explicitly set to NULL (no match).")
print(f"  Final UNIQ_IDs with values: {sorted(uniq_lookup.keys())}")
```

---

## Cell 15 — Step 9: Clean Up Temporary Datasets

Delete all intermediate datasets (zonal stats table, raster domain polygon, log-transformed raster) and print the final summary.

```python
print("\nSTEP 9: Cleaning up...")
delete_if_exists(zs_table)
delete_if_exists(raster_domain)
delete_if_exists(log_raster_path)

print(f"""
✅ ALL STEPS COMPLETE
   Values assigned to : {poly_3174}
   Join key used      : UNIQ_ID (OBJECTID-based, guaranteed unique)
   UPKh_krig          : Geometric mean — 10^(mean(log10(HK))) per polygon
   MidKh_krig         : UPKh_krig / 10
   Updated            : {updated} polygons
   NULL               : {nulled} polygons (outside raster or insufficient overlap)
   Overlap threshold  : {MIN_OVERLAP_FRACTION*100:.0f}%
""")
```


In [50]:
# ============================================================
# IMPORTS
# ============================================================
import os
from arcpy.sa import Con, IsNull, Raster, Log10
from collections import defaultdict, Counter

# ============================================================
# STEP 0. Helper functions
# ============================================================
def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def ensure_field(fc, field_name, field_type):
    existing = [f.name for f in arcpy.ListFields(fc)]
    if field_name not in existing:
        arcpy.management.AddField(fc, field_name, field_type)

# ============================================================
# STEP 1. Reproject POLYGON to EPSG:3174
# ============================================================
sr_3174 = arcpy.SpatialReference(3174)

print("STEP 1: Reprojecting polygon to EPSG:3174...")

poly_sr = arcpy.Describe(final_fc).spatialReference
if poly_sr.factoryCode != 3174:
    print(f"  ⚠️  Polygon SR is {poly_sr.name} — reprojecting...")
    poly_3174 = os.path.join(gdb, "poly_3174")
    delete_if_exists(poly_3174)
    arcpy.management.Project(final_fc, poly_3174, sr_3174)
    print(f"  ✅ Polygon reprojected: {poly_3174}")
else:
    poly_3174 = final_fc
    print(f"  ✅ Polygon already in EPSG:3174.")

# ============================================================
# STEP 2. Reproject RASTER to EPSG:3174
# ============================================================
print("\nSTEP 2: Reprojecting raster to EPSG:3174...")

ras_sr = arcpy.Describe(MIKrig).spatialReference
if ras_sr.factoryCode != 3174:
    print(f"  ⚠️  Raster SR is {ras_sr.name} — reprojecting...")
    raster_3174 = os.path.join(gdb, "MIKrig_3174")
    delete_if_exists(raster_3174)
    arcpy.management.ProjectRaster(
        MIKrig, raster_3174, sr_3174, "BILINEAR"
    )
    MIKrig_work = raster_3174
    print(f"  ✅ Raster reprojected: {raster_3174}")
else:
    MIKrig_work = MIKrig
    print(f"  ✅ Raster already in EPSG:3174.")

# ============================================================
# STEP 3. Build raster domain polygon
# ============================================================
print("\nSTEP 3: Building raster domain polygon...")

ras_obj    = Raster(MIKrig_work)
valid_mask = Con(~IsNull(ras_obj), 1)

raster_domain_raw = os.path.join(gdb, "temp_domain_raw")
raster_domain     = os.path.join(gdb, "temp_domain")
delete_if_exists(raster_domain_raw)
delete_if_exists(raster_domain)

arcpy.conversion.RasterToPolygon(
    valid_mask, raster_domain_raw,
    "NO_SIMPLIFY", "Value", "SINGLE_OUTER_PART"
)

arcpy.management.Dissolve(raster_domain_raw, raster_domain)
delete_if_exists(raster_domain_raw)

print(f"  ✅ Raster domain polygon created.")

# ============================================================
# Add unique join key BEFORE clipping
# ============================================================
print("\nAdding unique join key to polygon layer...")
ensure_field(poly_3174, "UNIQ_ID", "LONG")
arcpy.management.CalculateField(
    poly_3174, "UNIQ_ID",
    f"!{arcpy.Describe(poly_3174).OIDFieldName}!",
    "PYTHON3"
)
print(f"  ✅ UNIQ_ID populated from OBJECTID — guaranteed unique per row.")

uniq_counts = Counter()
with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID"]) as cur:
    for row in cur:
        uniq_counts[row[0]] += 1
print(f"  Total rows     : {sum(uniq_counts.values())}")
print(f"  Unique UNIQ_ID : {len(uniq_counts)}")
print(f"  Duplicates     : {sum(1 for c in uniq_counts.values() if c > 1)}")

# Store original areas for sliver detection later
orig_areas = {}
with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID", "SHAPE@AREA"]) as cur:
    for uniq_id, area in cur:
        orig_areas[uniq_id] = area

# ============================================================
# STEP 4. Clip polygon to raster domain + remove slivers
# ============================================================
print("\nSTEP 4: Clipping polygon to raster domain...")

clipped_fc_raw = os.path.join(gdb, "surfacegeo_HK_Krig_clipped_raw")
clipped_fc     = os.path.join(gdb, "surfacegeo_HK_Krig_clipped")
delete_if_exists(clipped_fc_raw)
delete_if_exists(clipped_fc)

arcpy.analysis.Clip(poly_3174, raster_domain, clipped_fc_raw)

raw_count   = int(arcpy.management.GetCount(clipped_fc_raw).getOutput(0))
total_count = int(arcpy.management.GetCount(poly_3174).getOutput(0))
print(f"  Raw clip result: {raw_count} features from {total_count} originals.")

# --- Remove slivers: keep largest fragment per UNIQ_ID,
#     and only if clipped area >= 5% of original area ---
print("  Removing sliver fragments...")

MIN_OVERLAP_FRACTION = 0.05  # 5% minimum overlap required

oid_field_raw = arcpy.Describe(clipped_fc_raw).OIDFieldName

# Collect all fragments per UNIQ_ID
uniq_fragments = defaultdict(list)
with arcpy.da.SearchCursor(
        clipped_fc_raw, [oid_field_raw, "UNIQ_ID", "SHAPE@AREA"]) as cur:
    for oid, uniq_id, area in cur:
        uniq_fragments[uniq_id].append((oid, area))

# Decide which OIDs to keep
keep_oids = set()
removed_slivers = []
for uniq_id, fragments in uniq_fragments.items():
    total_clip_area = sum(a for _, a in fragments)
    orig_area = orig_areas.get(uniq_id, 0)

    if orig_area > 0 and (total_clip_area / orig_area) < MIN_OVERLAP_FRACTION:
        removed_slivers.append(uniq_id)
        continue

    best_oid = max(fragments, key=lambda x: x[1])[0]
    keep_oids.add(best_oid)

# Delete fragments we don't want
all_oids_raw = set()
with arcpy.da.SearchCursor(clipped_fc_raw, [oid_field_raw]) as cur:
    for row in cur:
        all_oids_raw.add(row[0])

remove_oids = all_oids_raw - keep_oids

if remove_oids:
    lyr = "clipped_raw_lyr"
    arcpy.management.MakeFeatureLayer(clipped_fc_raw, lyr)
    where = f"{oid_field_raw} IN ({','.join(str(o) for o in remove_oids)})"
    arcpy.management.SelectLayerByAttribute(lyr, "NEW_SELECTION", where)
    removed_count = int(arcpy.management.GetCount(lyr).getOutput(0))
    arcpy.management.DeleteRows(lyr)
    arcpy.management.Delete(lyr)
    print(f"  ✅ Removed {removed_count} sliver/tiny-overlap fragments.")
    if removed_slivers:
        print(f"     UNIQ_IDs removed (< {MIN_OVERLAP_FRACTION*100:.0f}% overlap): "
              f"{sorted(removed_slivers)}")

arcpy.management.CopyFeatures(clipped_fc_raw, clipped_fc)
delete_if_exists(clipped_fc_raw)

clipped_count = int(arcpy.management.GetCount(clipped_fc).getOutput(0))
print(f"  ✅ {clipped_count} of {total_count} polygons kept after sliver removal.")
print(f"     {total_count - clipped_count} polygons removed.")

clipped_fields = [f.name for f in arcpy.ListFields(clipped_fc)]
if "UNIQ_ID" not in clipped_fields:
    raise RuntimeError("UNIQ_ID was not carried through the clip — cannot join.")
print(f"  ✅ UNIQ_ID confirmed present in clipped FC.")

clipped_uniq_ids = set()
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID"]) as cur:
    for row in cur:
        clipped_uniq_ids.add(row[0])
print(f"  Unique UNIQ_IDs in clipped: {len(clipped_uniq_ids)}")
print(f"  Clipped UNIQ_IDs: {sorted(clipped_uniq_ids)}")

uniq_clipped = Counter()
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID"]) as cur:
    for row in cur:
        uniq_clipped[row[0]] += 1
dups = {k: v for k, v in uniq_clipped.items() if v > 1}
if dups:
    print(f"  ⚠️  Duplicate UNIQ_IDs in clipped (slivers): {sorted(dups.keys())}")

# ============================================================
# STEP 5. Add ZONE_ID, UPKh_krig, MidKh_krig to clipped FC
# ============================================================
print("\nSTEP 5: Adding fields to clipped feature class...")

ensure_field(clipped_fc, "ZONE_ID",    "LONG")
ensure_field(clipped_fc, "UPKh_krig",  "DOUBLE")
ensure_field(clipped_fc, "MidKh_krig", "DOUBLE")

oid_field = arcpy.Describe(clipped_fc).OIDFieldName
arcpy.management.CalculateField(clipped_fc, "ZONE_ID", f"!{oid_field}!", "PYTHON3")

print(f"  ✅ ZONE_ID populated from {oid_field}.")
print(f"  ✅ UPKh_krig and MidKh_krig added.")

# ============================================================
# STEP 5b. Create log10-transformed raster
# ============================================================
print("\nSTEP 5b: Creating log10-transformed raster...")

ras_work = Raster(MIKrig_work)
log_raster_path = os.path.join(gdb, "temp_log10_HK")
delete_if_exists(log_raster_path)

# Only take log of valid positive cells
log_raster = Con(ras_work > 0, Log10(ras_work))
log_raster.save(log_raster_path)
print(f"  ✅ Log10(HK) raster created.")

# ============================================================
# STEP 6. Zonal Statistics (MEAN) on log10(HK) raster
# ============================================================
print("\nSTEP 6: Running Zonal Statistics (MEAN on log10 HK) on clipped polygons...")

arcpy.env.snapRaster = MIKrig_work
arcpy.env.cellSize   = MIKrig_work

zs_table = os.path.join(gdb, "temp_zs_krig")
delete_if_exists(zs_table)

try:
    arcpy.sa.ZonalStatisticsAsTable(
        clipped_fc, "ZONE_ID",
        log_raster_path, zs_table,
        "DATA", "MEAN"
    )
    if not arcpy.Exists(zs_table):
        raise RuntimeError("Output table not found after ZonalStatisticsAsTable.")
    zs_count = int(arcpy.management.GetCount(zs_table).getOutput(0))
    print(f"  ✅ Zonal Statistics complete — {zs_count} rows returned.")
except Exception as e:
    print(f"  ❌ ZonalStatisticsAsTable failed: {e}")
    print(f"     arcpy messages:\n{arcpy.GetMessages()}")
    raise

arcpy.env.snapRaster = None
arcpy.env.cellSize   = None

# ============================================================
# STEP 7. Back-transform and write geometric mean to clipped FC
# ============================================================
print("\nSTEP 7: Writing geometric mean HK values to clipped FC...")

zs_fields = [f.name for f in arcpy.ListFields(zs_table)]
print(f"  ZonalStats table fields: {zs_fields}")

if "MEAN" not in zs_fields:
    raise RuntimeError(f"MEAN field not found. Available: {zs_fields}")

# ZONE_ID → geometric mean HK = 10^(mean of log10(HK))
zs_dict = {}
with arcpy.da.SearchCursor(zs_table, ["ZONE_ID", "MEAN", "COUNT"]) as cur:
    for row in cur:
        zone_id, mean_log_val, count_val = row
        if mean_log_val is not None and count_val >= 1:
            zs_dict[zone_id] = 10 ** mean_log_val

print(f"  Valid ZonalStats results: {len(zs_dict)}")

updated = 0
with arcpy.da.UpdateCursor(
        clipped_fc, ["ZONE_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        zone_id = row[0]
        if zone_id in zs_dict:
            up_val = zs_dict[zone_id]
            row[1] = up_val
            row[2] = up_val / 10.0
            updated += 1
            cursor.updateRow(row)

print(f"  ✅ UPKh_krig (geometric mean) updated in {updated} clipped polygons.")
print(f"  ✅ MidKh_krig = UPKh_krig / 10 in {updated} clipped polygons.")

# ============================================================
# STEP 7b. Fill NULL polygons with nearest raster value
# ============================================================
print("\nSTEP 7b: Filling NULL UPKh_krig polygons...")

from arcpy.sa import Nibble

null_zone_ids = set()
with arcpy.da.SearchCursor(clipped_fc, ["ZONE_ID", "UPKh_krig"]) as cur:
    for row in cur:
        if row[1] is None:
            null_zone_ids.add(row[0])

print(f"  NULL polygons inside raster domain: {len(null_zone_ids)}")

if len(null_zone_ids) > 0:
    null_lyr = "null_polys_lyr"
    arcpy.management.MakeFeatureLayer(clipped_fc, null_lyr)
    null_ids_str = ", ".join(str(z) for z in null_zone_ids)
    arcpy.management.SelectLayerByAttribute(
        null_lyr, "NEW_SELECTION", f"ZONE_ID IN ({null_ids_str})")

    null_centroids = os.path.join(gdb, "temp_null_centroids")
    delete_if_exists(null_centroids)
    arcpy.management.FeatureToPoint(null_lyr, null_centroids, "INSIDE")
    arcpy.management.Delete(null_lyr)

    print(f"  ✅ {int(arcpy.management.GetCount(null_centroids).getOutput(0))} "
          f"NULL centroids created.")

    # Nibble from the log10 raster so filled values are in log space
    print("  Filling raster NoData holes with Nibble...")
    ras_log     = Raster(log_raster_path)
    valid_cells = Con(~IsNull(ras_log), 1)

    filled_raster = os.path.join(gdb, "temp_filled_raster")
    delete_if_exists(filled_raster)

    nibbled = arcpy.sa.Nibble(
        ras_log, valid_cells, "DATA_ONLY", "PROCESS_NODATA")
    nibbled.save(filled_raster)
    print(f"  ✅ Filled raster saved.")

    sample_null = os.path.join(gdb, "temp_sample_null")
    delete_if_exists(sample_null)
    arcpy.sa.Sample(
        filled_raster, null_centroids, sample_null, "NEAREST", "ZONE_ID")

    sample_fields = [f.name for f in arcpy.ListFields(sample_null)]
    link_field       = sample_fields[1]
    raster_val_field = sample_fields[-1]

    centroid_oid_field   = arcpy.Describe(null_centroids).OIDFieldName
    centroid_oid_to_zone = {}
    with arcpy.da.SearchCursor(null_centroids,
                               [centroid_oid_field, "ZONE_ID"]) as cur:
        for row in cur:
            centroid_oid_to_zone[row[0]] = row[1]

    filled_values = {}
    with arcpy.da.SearchCursor(sample_null,
                               [link_field, raster_val_field]) as cur:
        for row in cur:
            raw_oid, val = row
            zone_id = (
                centroid_oid_to_zone.get(raw_oid) or
                centroid_oid_to_zone.get(
                    int(raw_oid) if raw_oid is not None else None) or
                centroid_oid_to_zone.get(str(raw_oid))
            )
            if zone_id is not None and val is not None:
                # Back-transform from log10 space
                filled_values[zone_id] = 10 ** val

    print(f"  ✅ Nearest values found for {len(filled_values)} of "
          f"{len(null_zone_ids)} NULL polygons.")

    fill_count = 0
    with arcpy.da.UpdateCursor(
            clipped_fc, ["ZONE_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
        for row in cursor:
            zone_id = row[0]
            if zone_id in filled_values:
                up_val = filled_values[zone_id]
                row[1] = up_val
                row[2] = up_val / 10.0
                fill_count += 1
                cursor.updateRow(row)

    print(f"  ✅ {fill_count} NULL polygons filled.")

    still_null = null_zone_ids - set(filled_values.keys())
    if still_null:
        print(f"  ⚠️  {len(still_null)} polygons could not be filled: "
              f"{sorted(still_null)}")

    delete_if_exists(null_centroids)
    delete_if_exists(sample_null)
    delete_if_exists(filled_raster)
else:
    print("  ✅ No NULL polygons — nothing to fill.")

# ============================================================
# STEP 7c. Diagnostic
# ============================================================
print("\n  --- DIAGNOSTIC: UNIQ_IDs with non-NULL UPKh_krig in clipped FC ---")
valued_uniq_ids = set()
null_uniq_ids   = set()
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID", "UPKh_krig"]) as cur:
    for row in cur:
        if row[1] is not None:
            valued_uniq_ids.add(row[0])
        else:
            null_uniq_ids.add(row[0])
print(f"  UNIQ_IDs with values : {sorted(valued_uniq_ids)}")
print(f"  UNIQ_IDs still NULL  : {sorted(null_uniq_ids)}")

# ============================================================
# STEP 8. Join values back to original polygon using UNIQ_ID
#         STRICT: only clipped polygons with values get transferred
#         ALL others explicitly set to NULL — no exceptions
# ============================================================
print("\nSTEP 8: Assigning values to original polygon using UNIQ_ID...")

ensure_field(poly_3174, "UPKh_krig",  "DOUBLE")
ensure_field(poly_3174, "MidKh_krig", "DOUBLE")

# Build lookup from clipped FC — only non-NULL values
uniq_lookup = {}
sliver_ids  = []

with arcpy.da.SearchCursor(
        clipped_fc, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cur:
    for row in cur:
        uniq_id, up_val, mid_val = row
        if uniq_id is None or up_val is None:
            continue
        if uniq_id in uniq_lookup:
            sliver_ids.append(uniq_id)
            if up_val > uniq_lookup[uniq_id][0]:
                uniq_lookup[uniq_id] = (up_val, up_val / 10.0)
        else:
            uniq_lookup[uniq_id] = (up_val, up_val / 10.0)

print(f"  Unique UNIQ_IDs with values : {len(uniq_lookup)}")
print(f"  UNIQ_IDs getting values: {sorted(uniq_lookup.keys())}")
if sliver_ids:
    print(f"  Slivers resolved (MAX kept) : {sorted(set(sliver_ids))}")

# EVERY row gets written: matched → values, unmatched → NULL
updated = 0
nulled  = 0
with arcpy.da.UpdateCursor(
        poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if uniq_id in uniq_lookup:
            row[1] = uniq_lookup[uniq_id][0]   # UPKh_krig
            row[2] = uniq_lookup[uniq_id][1]   # MidKh_krig
            updated += 1
        else:
            row[1] = None   # force NULL
            row[2] = None   # force NULL
            nulled += 1
        cursor.updateRow(row)

print(f"  ✅ {updated} polygons received values (matched on UNIQ_ID).")
print(f"  ✅ {nulled} polygons explicitly set to NULL (no match).")
print(f"  Final UNIQ_IDs with values: {sorted(uniq_lookup.keys())}")

# ============================================================
# STEP 9. Clean up
# ============================================================
print("\nSTEP 9: Cleaning up...")
delete_if_exists(zs_table)
delete_if_exists(raster_domain)
delete_if_exists(log_raster_path)

print(f"""
✅ ALL STEPS COMPLETE
   Values assigned to : {poly_3174}
   Join key used      : UNIQ_ID (OBJECTID-based, guaranteed unique)
   UPKh_krig          : Geometric mean — 10^(mean(log10(HK))) per polygon
   MidKh_krig         : UPKh_krig / 10
   Updated            : {updated} polygons
   NULL               : {nulled} polygons (outside raster or insufficient overlap)
   Overlap threshold  : {MIN_OVERLAP_FRACTION*100:.0f}%
""")

STEP 1: Reprojecting polygon to EPSG:3174...
  ⚠️  Polygon SR is NAD_1983_UTM_Zone_17N — reprojecting...
  ✅ Polygon reprojected: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_join_clip.gdb\poly_3174

STEP 2: Reprojecting raster to EPSG:3174...
  ✅ Raster already in EPSG:3174.

STEP 3: Building raster domain polygon...
  ✅ Raster domain polygon created.

Adding unique join key to polygon layer...
  ✅ UNIQ_ID populated from OBJECTID — guaranteed unique per row.
  Total rows     : 74
  Unique UNIQ_ID : 74
  Duplicates     : 0

STEP 4: Clipping polygon to raster domain...
  Raw clip result: 39 features from 74 originals.
  Removing sliver fragments...
  ✅ Removed 15 sliver/tiny-overlap fragments.
     UNIQ_IDs removed (< 5% overlap): [3, 5, 7, 8, 9, 13, 16, 17, 23, 25, 49, 53, 57, 61, 74]
  ✅ 24 of 74 polygons kept after sliver removal.
     50 polygons removed.
  ✅ UNIQ_ID confirmed present in clipped FC.
  Unique UNIQ_IDs in clipped: 24
  Clipped UNIQ_IDs: [12, 

## 16 Build Lookup Table from Michigan Polygons
Read all polygons that have UPKh_krig values (Michigan polygons) and create a lookup table showing each unique Description with its associated POLYID, UPKh_krig, and MidKh_krig.

In [51]:
import pandas as pd

print("Building lookup table from Michigan polygons...")

# Read all polygons with values (Michigan)
michigan_records = []
with arcpy.da.SearchCursor(
        poly_3174, 
        ["UNIQ_ID", "POLYID", "Description", "UPKh_krig", "MidKh_krig", "SHAPE@"]) as cur:
    for row in cur:
        uniq_id, polyid, desc, up_val, mid_val, shape = row
        if up_val is not None:
            michigan_records.append({
                "UNIQ_ID":    uniq_id,
                "POLYID":     polyid,
                "Description": desc,
                "UPKh_krig":  up_val,
                "MidKh_krig": mid_val,
                "CENTROID_X": shape.centroid.X,
                "CENTROID_Y": shape.centroid.Y
            })

michigan_df = pd.DataFrame(michigan_records)
print(f"  ✅ {len(michigan_df)} Michigan polygons with HK values.")
print(f"  Unique Descriptions: {michigan_df['Description'].nunique()}")
print()

# Display the lookup table
lookup = michigan_df[["POLYID", "Description", "UPKh_krig", "MidKh_krig"]].sort_values("Description")
print(lookup.to_string(index=False))

Building lookup table from Michigan polygons...
  ✅ 22 Michigan polygons with HK values.
  Unique Descriptions: 15

 POLYID                                                        Description  UPKh_krig  MidKh_krig
      1                    Alluvial sediments - Undifferentiated sediments   0.127503    0.012750
     28                          Eolian sediments, mostly dune sand, thick  20.113689    2.011369
     48                           Eolian sediments, mostly dune sand, thin   4.585282    0.458528
     48                           Eolian sediments, mostly dune sand, thin  11.432370    1.143237
     27                       Glacial till sediments, mostly clayey, thick  18.044275    1.804427
     19                Glacial till sediments, mostly silty, discontinuous   4.653132    0.465313
     22                        Glacial till sediments, mostly silty, thick  22.669901    2.266990
     22                        Glacial till sediments, mostly silty, thick   4.897357    0.489736
  

# Cell 16b — Fuzzy Description Matching

In [56]:
# ============================================================
# Cell 16b — Fuzzy Description Matching
# ============================================================
# Strategy: strip the thickness qualifier (thick/thin/discontinuous/veneer/blanket)
# and match on the base description (sediment type + grain size)

import re

def get_base_description(desc):
    """
    Strip thickness/coverage qualifiers from the end of a description
    to get the base sediment type for fuzzy matching.
    e.g. 'Glacial till sediments, mostly clayey, thick' 
       → 'Glacial till sediments, mostly clayey'
    """
    if desc is None:
        return None
    
    # Remove trailing qualifiers
    qualifiers = [
        r',\s*(thick|thin|discontinuous)$',
        r'\s*-\s*(Blanket|Veneer|Moraine complex)$',
        r'\s*-\s*(Undifferentiated sediments|Undifferentiated deposits|Undifferentiated)$',
        r'\s*-\s*(Ice-contact sediments|Outwash plain sediments)$',
        r'\s*-\s*(Littoral and nearshore sediments|Offshore sediments)$',
    ]
    
    base = desc.strip()
    for pattern in qualifiers:
        base = re.sub(pattern, '', base, flags=re.IGNORECASE)
    
    return base.strip()

# Build base description → Michigan values lookup
michigan_base_lookup = defaultdict(list)
for _, row in michigan_df.iterrows():
    base = get_base_description(row["Description"])
    michigan_base_lookup[base].append({
        "POLYID":      row["POLYID"],
        "Description": row["Description"],
        "UPKh_krig":   row["UPKh_krig"],
        "MidKh_krig":  row["MidKh_krig"],
        "x":           row["CENTROID_X"],
        "y":           row["CENTROID_Y"]
    })

# Show the base descriptions available in Michigan
print("Michigan base descriptions available for fuzzy matching:")
print("=" * 70)
for base, entries in sorted(michigan_base_lookup.items()):
    descs = set(e["Description"] for e in entries)
    vals  = [e["UPKh_krig"] for e in entries]
    print(f"  Base: '{base}'")
    print(f"    Full descriptions: {sorted(descs)}")
    print(f"    UPKh_krig values : {[round(v, 4) for v in vals]}")
    print()

# Now match each unmatched NULL polygon
print("\nFuzzy matching results for unmatched NULL polygons:")
print("=" * 70)

fuzzy_assignments = {}

for _, null_row in null_df.iterrows():
    uid  = null_row["UNIQ_ID"]
    desc = null_row["Description"]
    
    # Skip if already matched exactly
    if uid in assignments:
        continue
    
    base = get_base_description(desc)
    
    if base in michigan_base_lookup:
        # Find nearest Michigan polygon with same base description
        null_x = null_row["CENTROID_X"]
        null_y = null_row["CENTROID_Y"]
        
        best_dist  = float("inf")
        best_match = None
        
        for mi_poly in michigan_base_lookup[base]:
            dist = np.sqrt((null_x - mi_poly["x"])**2 + (null_y - mi_poly["y"])**2)
            if dist < best_dist:
                best_dist  = dist
                best_match = mi_poly
        
        if best_match is not None:
            fuzzy_assignments[uid] = {
                "UPKh_krig":       best_match["UPKh_krig"],
                "MidKh_krig":      best_match["MidKh_krig"],
                "source_POLYID":   best_match["POLYID"],
                "source_desc":     best_match["Description"],
                "distance_m":      best_dist
            }
            print(f"  UNIQ_ID {uid:>3} | POLYID {null_row['POLYID']:>3}")
            print(f"    NULL desc   : '{desc}'")
            print(f"    Matched to  : '{best_match['Description']}' (POLYID {best_match['POLYID']})")
            print(f"    UPKh_krig   : {best_match['UPKh_krig']:.4f}")
            print(f"    Distance    : {best_dist:,.0f} m")
            print()
    else:
        print(f"  UNIQ_ID {uid:>3} | POLYID {null_row['POLYID']:>3}")
        print(f"    NULL desc   : '{desc}'")
        print(f"    ❌ No fuzzy match found in Michigan")
        print()

matched_fuzzy = len(fuzzy_assignments)
still_unmatched = len(null_df) - len(assignments) - matched_fuzzy
print(f"\nSummary:")
print(f"  Exact matches  : {len(assignments)}")
print(f"  Fuzzy matches  : {matched_fuzzy}")
print(f"  Still unmatched: {still_unmatched}")

Michigan base descriptions available for fuzzy matching:
  Base: 'Alluvial sediments'
    Full descriptions: ['Alluvial sediments - Undifferentiated sediments']
    UPKh_krig values : [0.1275]

  Base: 'Eolian sediments, mostly dune sand'
    Full descriptions: ['Eolian sediments, mostly dune sand, thick', 'Eolian sediments, mostly dune sand, thin']
    UPKh_krig values : [11.4324, 20.1137, 4.5853]

  Base: 'Glacial till sediments, mostly clayey'
    Full descriptions: ['Glacial till sediments, mostly clayey, thick']
    UPKh_krig values : [18.0443]

  Base: 'Glacial till sediments, mostly silty'
    Full descriptions: ['Glacial till sediments, mostly silty, discontinuous', 'Glacial till sediments, mostly silty, thick', 'Glacial till sediments, mostly silty, thin']
    UPKh_krig values : [4.6531, 22.6699, 13.2909, 4.8974, 4.7017]

  Base: 'Glaciofluvial ice-contact sediments, mostly sand and gravel'
    Full descriptions: ['Glaciofluvial ice-contact sediments, mostly sand and gravel, t

In [57]:
# ============================================================
# Cell 16c — Fuzzy Matching with fallback field display
# ============================================================
# For unmatched polygons, show their UPKh_m_d and MidKh_m_d values
# so you can decide what to assign manually

assignments = {}

print("\nFuzzy matching results for unmatched NULL polygons:")
print("=" * 70)

fuzzy_assignments = {}

# Read UPKh_m_d and MidKh_m_d for all polygons
existing_vals = {}
with arcpy.da.SearchCursor(
        poly_3174, ["UNIQ_ID", "UPKh_m_d", "MidKh_m_d"]) as cur:
    for row in cur:
        existing_vals[row[0]] = (row[1], row[2])

for _, null_row in null_df.iterrows():
    uid  = null_row["UNIQ_ID"]
    desc = null_row["Description"]
    
    if uid in assignments:
        continue
    
    base = get_base_description(desc)
    
    if base in michigan_base_lookup:
        null_x = null_row["CENTROID_X"]
        null_y = null_row["CENTROID_Y"]
        
        best_dist  = float("inf")
        best_match = None
        
        for mi_poly in michigan_base_lookup[base]:
            dist = np.sqrt((null_x - mi_poly["x"])**2 + (null_y - mi_poly["y"])**2)
            if dist < best_dist:
                best_dist  = dist
                best_match = mi_poly
        
        if best_match is not None:
            fuzzy_assignments[uid] = {
                "UPKh_krig":       best_match["UPKh_krig"],
                "MidKh_krig":      best_match["MidKh_krig"],
                "source_POLYID":   best_match["POLYID"],
                "source_desc":     best_match["Description"],
                "distance_m":      best_dist
            }
            print(f"  UNIQ_ID {uid:>3} | POLYID {null_row['POLYID']}")
            print(f"    NULL desc   : '{desc}'")
            print(f"    ✅ Matched to  : '{best_match['Description']}' (POLYID {best_match['POLYID']})")
            print(f"    UPKh_krig   : {best_match['UPKh_krig']:.4f}")
            print(f"    Distance    : {best_dist:,.0f} m")
            print()
    else:
        up_md, mid_md = existing_vals.get(uid, (None, None))
        print(f"  UNIQ_ID {uid:>3} | POLYID {null_row['POLYID']}")
        print(f"    NULL desc    : '{desc}'")
        print(f"    ❌ No fuzzy match found in Michigan")
        print(f"    UPKh_m_d     : {up_md}")
        print(f"    MidKh_m_d    : {mid_md}")
        print()

matched_fuzzy = len(fuzzy_assignments)
still_unmatched = len(null_df) - len(assignments) - matched_fuzzy
print(f"\nSummary:")
print(f"  Exact matches  : {len(assignments)}")
print(f"  Fuzzy matches  : {matched_fuzzy}")
print(f"  Still unmatched: {still_unmatched}")

# Summary table of unmatched with their existing values
print(f"\n\nUnmatched polygons with existing UPKh_m_d and MidKh_m_d:")
print(f"  {'UNIQ_ID':>8}  {'POLYID':>8}  {'Description':<55}  {'UPKh_m_d':>10}  {'MidKh_m_d':>10}")
print(f"  {'─'*8}  {'─'*8}  {'─'*55}  {'─'*10}  {'─'*10}")

for _, null_row in null_df.iterrows():
    uid = null_row["UNIQ_ID"]
    if uid not in fuzzy_assignments and uid not in assignments:
        up_md, mid_md = existing_vals.get(uid, (None, None))
        up_str  = f"{up_md:.4f}" if up_md is not None else "NULL"
        mid_str = f"{mid_md:.4f}" if mid_md is not None else "NULL"
        print(f"  {uid:>8}  {null_row['POLYID']:>8}  {str(null_row['Description']):<55}  "
              f"{up_str:>10}  {mid_str:>10}")


Fuzzy matching results for unmatched NULL polygons:
  UNIQ_ID   1 | POLYID 29.0
    NULL desc    : 'Coastal zone sediments, mostly medium-grained'
    ❌ No fuzzy match found in Michigan
    UPKh_m_d     : 29.030399999999997
    MidKh_m_d    : 29.030399999999997

  UNIQ_ID   2 | POLYID 45.0
    NULL desc    : 'Colluvial sediments and residual material'
    ❌ No fuzzy match found in Michigan
    UPKh_m_d     : 87.26400000000001
    MidKh_m_d    : 87.26400000000001

  UNIQ_ID   3 | POLYID 5.0
    NULL desc    : 'Glaciolacustrine sediments - Offshore sediments'
    ❌ No fuzzy match found in Michigan
    UPKh_m_d     : 5.00256
    MidKh_m_d    : 60.48

  UNIQ_ID   4 | POLYID 8.0
    NULL desc    : 'Lacustrine sediments - Littoral and nearshore sediments'
    ❌ No fuzzy match found in Michigan
    UPKh_m_d     : 31.0176
    MidKh_m_d    : 31.0176

  UNIQ_ID   5 | POLYID 12.0
    NULL desc    : 'Glacial sediments - Blanket'
    ❌ No fuzzy match found in Michigan
    UPKh_m_d     : 0.19958399

## Identify NULL Polygons and Their Descriptions
Read all polygons that are still NULL. Show their descriptions and flag which ones have a matching description in Michigan.

In [52]:
print("\nIdentifying NULL polygons and matching descriptions...")

null_records = []
with arcpy.da.SearchCursor(
        poly_3174,
        ["UNIQ_ID", "POLYID", "Description", "UPKh_krig", "SHAPE@"]) as cur:
    for row in cur:
        uniq_id, polyid, desc, up_val, shape = row
        if up_val is None:
            null_records.append({
                "UNIQ_ID":     uniq_id,
                "POLYID":      polyid,
                "Description": desc,
                "CENTROID_X":  shape.centroid.X,
                "CENTROID_Y":  shape.centroid.Y
            })

null_df = pd.DataFrame(null_records)
print(f"  ✅ {len(null_df)} NULL polygons found.")

# Check which descriptions have matches in Michigan
michigan_descs = set(michigan_df["Description"].dropna().unique())
null_df["Has_Match"] = null_df["Description"].isin(michigan_descs)

matched     = null_df[null_df["Has_Match"] == True]
not_matched = null_df[null_df["Has_Match"] == False]

print(f"\n  With matching Michigan description : {len(matched)}")
print(f"  No match in Michigan               : {len(not_matched)}")

if len(not_matched) > 0:
    print(f"\n  ⚠️  Unmatched descriptions:")
    for desc in sorted(not_matched["Description"].dropna().unique()):
        count = len(not_matched[not_matched["Description"] == desc])
        print(f"      - '{desc}' ({count} polygons)")


Identifying NULL polygons and matching descriptions...
  ✅ 52 NULL polygons found.

  With matching Michigan description : 5
  No match in Michigan               : 47

  ⚠️  Unmatched descriptions:
      - 'Alluvial sediments, thick' (2 polygons)
      - 'Alluvial sediments, thin' (1 polygons)
      - 'Bedrock - Undifferentiated' (2 polygons)
      - 'Coastal zone sediments, mostly medium-grained' (2 polygons)
      - 'Colluvial and alluvial sediments' (1 polygons)
      - 'Colluvial sediments and residual material' (1 polygons)
      - 'Colluvial sediments, discontinuous' (1 polygons)
      - 'Colluvial sediments, thin' (1 polygons)
      - 'Glacial sediments - Blanket' (2 polygons)
      - 'Glacial sediments - Moraine complex' (2 polygons)
      - 'Glacial sediments - Veneer' (2 polygons)
      - 'Glacial till sediments, mostly clayey, discontinuous' (2 polygons)
      - 'Glacial till sediments, mostly clayey, thin' (2 polygons)
      - 'Glacial till sediments, mostly sandy, discont

In [55]:
import numpy as np

print("\nAssigning values from nearest matching Michigan polygon...")

# Build spatial index: Description → list of (x, y, UPKh, MidKh)
from collections import defaultdict

michigan_by_desc = defaultdict(list)
for _, row in michigan_df.iterrows():
    michigan_by_desc[row["Description"]].append({
        "POLYID":     row["POLYID"],
        "x":          row["CENTROID_X"],
        "y":          row["CENTROID_Y"],
        "UPKh_krig":  row["UPKh_krig"],
        "MidKh_krig": row["MidKh_krig"]
    })

# For each NULL polygon, find nearest Michigan polygon with same Description
assignments = {}  # UNIQ_ID → (UPKh_krig, MidKh_krig, source_POLYID, distance)

for _, null_row in null_df.iterrows():
    desc = null_row["Description"]
    if desc not in michigan_by_desc:
        continue

    null_x = null_row["CENTROID_X"]
    null_y = null_row["CENTROID_Y"]

    best_dist = float("inf")
    best_match = None

    for mi_poly in michigan_by_desc[desc]:
        dist = np.sqrt((null_x - mi_poly["x"])**2 + (null_y - mi_poly["y"])**2)
        if dist < best_dist:
            best_dist = dist
            best_match = mi_poly

    if best_match is not None:
        assignments[null_row["UNIQ_ID"]] = {
            "UPKh_krig":    best_match["UPKh_krig"],
            "MidKh_krig":   best_match["MidKh_krig"],
            "source_POLYID": best_match["POLYID"],
            "distance_m":   best_dist
        }

print(f"  ✅ {len(assignments)} NULL polygons matched to nearest Michigan polygon.")

# Show assignments
print(f"\n  {'UNIQ_ID':>8}  {'POLYID':>8}  {'Description':<30}  {'Source_POLYID':>13}  "
      f"{'UPKh_krig':>10}  {'MidKh_krig':>10}  {'Dist_m':>10}")
print(f"  {'─'*8}  {'─'*8}  {'─'*30}  {'─'*13}  {'─'*10}  {'─'*10}  {'─'*10}")

for _, null_row in null_df.iterrows():
    uid = null_row["UNIQ_ID"]
    if uid in assignments:
        a = assignments[uid]
        print(f"  {uid:>8}  {null_row['POLYID']:>8}  {str(null_row['Description']):<30}  "
              f"{a['source_POLYID']:>13}  {a['UPKh_krig']:>10.4f}  "
              f"{a['MidKh_krig']:>10.4f}  {a['distance_m']:>10.0f}")


Assigning values from nearest matching Michigan polygon...
  ✅ 5 NULL polygons matched to nearest Michigan polygon.

   UNIQ_ID    POLYID  Description                     Source_POLYID   UPKh_krig  MidKh_krig      Dist_m
  ────────  ────────  ──────────────────────────────  ─────────────  ──────────  ──────────  ──────────
        57      19.0  Glacial till sediments, mostly silty, discontinuous             19      4.6531      0.4653       48700
        61      27.0  Glacial till sediments, mostly clayey, thick             27     18.0443      1.8044      829657
        62      28.0  Eolian sediments, mostly dune sand, thick             28     20.1137      2.0114      390959
        65      38.0  Organic-rich muck and peat, thick             38     18.3849      1.8385      184900
        73      51.0  Proglacial sediments, mostly fine grained, thick             51      3.7286      0.3729      418663


## Write Matched Values to Original Polygon
Update the original polygon layer with the values from the nearest matching Michigan polygon. Only polygons with a description match get updated — all others remain NUL

In [53]:
print("\nWriting matched values to original polygon...")

filled = 0
with arcpy.da.UpdateCursor(
        poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if uniq_id in assignments and row[1] is None:
            row[1] = assignments[uniq_id]["UPKh_krig"]
            row[2] = assignments[uniq_id]["MidKh_krig"]
            filled += 1
            cursor.updateRow(row)

print(f"  ✅ {filled} NULL polygons filled from nearest Michigan match.")

# Final count
total_null = 0
total_filled = 0
with arcpy.da.SearchCursor(poly_3174, ["UPKh_krig"]) as cur:
    for row in cur:
        if row[0] is None:
            total_null += 1
        else:
            total_filled += 1

print(f"\n  Final summary:")
print(f"    Polygons with values : {total_filled}")
print(f"    Polygons still NULL  : {total_null}")

if total_null > 0:
    print(f"\n  ⚠️  {total_null} polygons remain NULL — no matching Description in Michigan.")
    print(f"      These will need manual assignment or a different matching strategy.")


Writing matched values to original polygon...


NameError: name 'assignments' is not defined